In [1]:
import os
# os.getcwd()
os.chdir("..")

In [2]:
os.listdir()

['.git',
 'README.md',
 '.ipynb_checkpoints',
 '.gitignore',
 'notebooks',
 'src',
 'requirements.txt',
 'data',
 'models',
 'ct_pathology_results_20251002_091909.xlsx',
 'ct_pathology_results_20251002_093331.xlsx',
 'ct_pathology_results_20251002_093732.xlsx',
 'ct_pathology_results_20251002_093856.xlsx',
 'ct_pathology_results_20251002_094312.xlsx',
 'ct_pathology_results_20251002_094708.xlsx',
 'ct_pathology_results_20251002_095023.xlsx',
 'ct_pathology_results_20251002_095231.xlsx',
 'ct_pathology_results_20251002_095321.xlsx',
 'ct_pathology_results_20251002_095638.xlsx']

In [3]:
# test_fixed_pipeline.py
import os
import tempfile
import zipfile
import numpy as np
import nibabel as nib
from pathlib import Path

def create_real_test_zip():
    """Создаём реальный тестовый NIfTI файл"""
    temp_dir = tempfile.mkdtemp()
    zip_path = os.path.join(temp_dir, "test_data.zip")
    
    with zipfile.ZipFile(zip_path, 'w') as zf:
        # Создаём настоящий NIfTI файл
        dummy_volume = np.random.randint(-1000, 1000, (64, 64, 32), dtype=np.int16)
        
        # Создаём NIfTI изображение с правильными header
        affine = np.eye(4)
        affine[:3, :3] *= [0.8, 0.8, 1.5]  # Real CT spacing
        
        img = nib.Nifti1Image(dummy_volume, affine)
        
        # Сохраняем временно
        nifti_path = os.path.join(temp_dir, "ct_scan.nii.gz")
        nib.save(img, nifti_path)
        
        # Добавляем в ZIP
        zf.write(nifti_path, "study_001/ct_scan.nii.gz")
    
    return zip_path

def test_fixed_pipeline():
    """Тест исправленного pipeline"""
    print("🧪 Testing FIXED Core Pipeline...")
    
    try:
        # Импорт исправленных модулей
        from src.pipeline.core_pipeline import CTPathologyPipeline
        from src.pipeline.data_models import PipelineConfig  # Нужно также создать
        
        # Конфигурация
        config = PipelineConfig(
            ctclip_checkpoint="models/CT_LiPro_v2.pt",
            catboost_model="models/catboost_pathology_classifier.cbm",
            device='cuda',
            max_workers=1,
            log_level='INFO'
        )
        
        # Создаём pipeline
        pipeline = CTPathologyPipeline(config)
        
        # Тестовые данные
        test_zip = create_real_test_zip()
        print(f"📦 Test ZIP: {test_zip}")
        
        # ГЛАВНЫЙ ТЕСТ
        excel_path, stats = pipeline.process_zip_archives([test_zip])
        
        print(f"✅ SUCCESS! Excel: {excel_path}")
        print(f"📊 Stats: {stats}")
        
        return True
        
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False

if __name__ == "__main__":
    success = test_fixed_pipeline()
    print(f"\n{'🎉 PIPELINE WORKS!' if success else '💥 STILL ISSUES'}")


🧪 Testing FIXED Core Pipeline...


/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-10-02 09:59:53.717111: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-02 09:59:53.766124: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-02 09:59:54.802255: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-10-02 09:59:55,786 - CTPa

📦 Test ZIP: /tmp/tmpsib447fc/test_data.zip


2025-10-02 10:00:02,194 - CTPathologyPipeline - INFO - 🔍 Сканирование: /tmp/ct_pipeline_3njl9gcz/archive_0
2025-10-02 10:00:02,195 - CTPathologyPipeline - INFO - 📁 Поиск NIfTI...
2025-10-02 10:00:02,197 - CTPathologyPipeline - INFO -   ✅ NIfTI: 1
2025-10-02 10:00:02,198 - CTPathologyPipeline - INFO - 📁 Поиск DICOM...
2025-10-02 10:00:02,199 - CTPathologyPipeline - INFO -   ✅ DICOM: 0
2025-10-02 10:00:02,200 - CTPathologyPipeline - INFO - 🎯 Всего: 1
2025-10-02 10:00:02,206 - CTPathologyPipeline - WARNING - ⚠️ Failed to read NIfTI metadata: 'numpy.ndarray' object has no attribute 'decode'
2025-10-02 10:00:02,207 - CTPathologyPipeline - INFO - ✅ Found 1 studies in /tmp/tmpsib447fc/test_data.zip
2025-10-02 10:00:02,208 - CTPathologyPipeline - INFO -   ✅ Found 1 studies
2025-10-02 10:00:02,209 - CTPathologyPipeline - INFO - 📊 Total studies discovered: 1
2025-10-02 10:00:02,209 - CTPathologyPipeline - INFO - 🔄 Sequential processing of 1 studies...
Processing studies:   0%|          | 0/1 [00

✅ SUCCESS! Excel: ct_pathology_results_20251002_100002.xlsx
📊 Stats: {'total_studies': 1, 'successful_studies': 0, 'failed_studies': 1, 'success_rate': 0.0, 'pathology_detected': 0, 'normal_studies': 1, 'pathology_rate': 0, 'total_processing_time': 2.0609488487243652, 'average_time_per_study': 0, 'min_processing_time': 0, 'max_processing_time': 0, 'studies_per_minute': 29.11280405485915}

🎉 PIPELINE WORKS!


In [15]:
# test_fixed_pipeline.py
import os
import tempfile
import zipfile
import numpy as np
import nibabel as nib
from pathlib import Path

def create_real_test_zip():
    """Создаём ZIP с ПРАВИЛЬНОЙ DICOM структурой"""
    temp_dir = tempfile.mkdtemp()
    zip_path = os.path.join(temp_dir, "test_data.zip")
    
    dicom_source = "/home/jupyter/datasphere/project/chest-ct-classification/data/dataset_subset/test_subset_small/dicom/1.2.643.5.1.13.13.12.2.77.8252.02120609031109060805091105001209"
    
    with zipfile.ZipFile(zip_path, 'w') as zf:
        # ✅ ПРАВИЛЬНО: Добавляем все .dcm файлы из папки
        if os.path.exists(dicom_source):
            for file in os.listdir(dicom_source):
                if file.endswith('.dcm'):
                    file_path = os.path.join(dicom_source, file)
                    # Структура: study_001/file.dcm
                    zf.write(file_path, f"study_001/{file}")
        else:
            # Fallback: создаём fake DICOM файлы
            for i in range(50):  # 50 слайсов
                fake_dicom_content = f"FAKE_DICOM_SLICE_{i:03d}"
                zf.writestr(f"study_001/slice_{i:03d}.dcm", fake_dicom_content)
    
    return zip_path

def test_fixed_pipeline():
    """Тест исправленного pipeline"""
    print("🧪 Testing FIXED Core Pipeline...")
    
    try:
        # Импорт исправленных модулей
        from src.pipeline.core_pipeline import CTPathologyPipeline
        from src.pipeline.data_models import PipelineConfig  # Нужно также создать
        
        # Конфигурация
        config = PipelineConfig(
            ctclip_checkpoint="models/CT_LiPro_v2.pt",
            catboost_model="models/catboost_pathology_classifier.cbm",
            device='cuda',
            max_workers=1,
            log_level='INFO'
        )
        
        # Создаём pipeline
        pipeline = CTPathologyPipeline(config)
        
        # Тестовые данные
        test_zip = create_real_test_zip()
        print(f"📦 Test ZIP: {test_zip}")
        
        # ГЛАВНЫЙ ТЕСТ
        excel_path, stats = pipeline.process_zip_archives([test_zip])
        
        print(f"✅ SUCCESS! Excel: {excel_path}")
        print(f"📊 Stats: {stats}")
        
        return True
        
    except Exception as e:
        print(f"❌ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False

if __name__ == "__main__":
    success = test_fixed_pipeline()
    print(f"\n{'🎉 PIPELINE WORKS!' if success else '💥 STILL ISSUES'}")

🧪 Testing FIXED Core Pipeline...
❌ FAILED: invalid syntax (core_pipeline.py, line 267)

💥 STILL ISSUES


Traceback (most recent call last):
  File "/tmp/ipykernel_3486/3229517635.py", line 38, in test_fixed_pipeline
    from src.pipeline.core_pipeline import CTPathologyPipeline
  File "/home/jupyter/work/resources/chest-ct-classification/src/pipeline/core_pipeline.py", line 267
    preprocessed = preprocess_nifti(volume_data.volume, mock_metadata, Volume=True=)
                                                                                  ^
SyntaxError: invalid syntax
